# Введение в MapReduce модель на Python


In [29]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

In [3]:
def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

Модель элемента данных

In [4]:
class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

In [5]:
input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

Функция RECORDREADER моделирует чтение элементов с диска или по сети.

In [6]:
def RECORDREADER():
  return [(u.id, u) for u in input_collection]

In [7]:
list(RECORDREADER())

[(0, User(id=0, age=55, social_contacts=20, gender='male')),
 (1, User(id=1, age=25, social_contacts=240, gender='female')),
 (2, User(id=2, age=25, social_contacts=500, gender='female')),
 (3, User(id=3, age=33, social_contacts=800, gender='female'))]

In [8]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

In [9]:
map_output = flatten(map(lambda x: MAP(*x), RECORDREADER()))
map_output = list(map_output) # materialize
map_output

[(25, User(id=1, age=25, social_contacts=240, gender='female')),
 (25, User(id=2, age=25, social_contacts=500, gender='female')),
 (33, User(id=3, age=33, social_contacts=800, gender='female'))]

In [10]:
def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

In [11]:
shuffle_output = groupbykey(map_output)
shuffle_output = list(shuffle_output)
shuffle_output

[(25,
  [User(id=1, age=25, social_contacts=240, gender='female'),
   User(id=2, age=25, social_contacts=500, gender='female')]),
 (33, [User(id=3, age=33, social_contacts=800, gender='female')])]

In [12]:
reduce_output = flatten(map(lambda x: REDUCE(*x), shuffle_output))
reduce_output = list(reduce_output)
reduce_output

[(25, 370.0), (33, 800.0)]

Все действия одним конвейером!

In [13]:
list(flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER()))))))

[(25, 370.0), (33, 800.0)]

# **MapReduce**
Выделим общую для всех пользователей часть системы в отдельную функцию высшего порядка. Это наиболее простая модель MapReduce, без учёта распределённого хранения данных.

Пользователь для решения своей задачи реализует RECORDREADER, MAP, REDUCE.

In [14]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

## Спецификация MapReduce



```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

mapreduce ((k1,v1)*) -> (k3,v3)*
groupby ((k2,v2)*) -> (k2,v2*)*
flatten (e2**) -> e2*

mapreduce .map(f).flatten.groupby(k2).map(g).flatten
```




# Примеры

## SQL

In [15]:
from typing import NamedTuple # requires python 3.6+
from typing import Iterator

class User(NamedTuple):
  id: int
  age: str
  social_contacts: int
  gender: str

input_collection = [
    User(id=0, age=55, gender='male', social_contacts=20),
    User(id=1, age=25, gender='female', social_contacts=240),
    User(id=2, age=25, gender='female', social_contacts=500),
    User(id=3, age=33, gender='female', social_contacts=800)
]

def MAP(_, row:NamedTuple):
  if (row.gender == 'female'):
    yield (row.age, row)

def REDUCE(age:str, rows:Iterator[NamedTuple]):
  sum = 0
  count = 0
  for row in rows:
    sum += row.social_contacts
    count += 1
  if (count > 0):
    yield (age, sum/count)
  else:
    yield (age, 0)

def RECORDREADER():
  return [(u.id, u) for u in input_collection]

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(25, 370.0), (33, 800.0)]

## Matrix-Vector multiplication

In [16]:
from typing import Iterator
import numpy as np

mat = np.ones((5,4))
vec = np.random.rand(4) # in-memory vector in all map tasks

def MAP(coordinates:(int, int), value:int):
  i, j = coordinates
  yield (i, value*vec[j])

def REDUCE(i:int, products:Iterator[NamedTuple]):
  sum = 0
  for p in products:
    sum += p
  yield (i, sum)

def RECORDREADER():
  for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
      yield ((i, j), mat[i,j])

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[(0, np.float64(2.516984896242605)),
 (1, np.float64(2.516984896242605)),
 (2, np.float64(2.516984896242605)),
 (3, np.float64(2.516984896242605)),
 (4, np.float64(2.516984896242605))]

## Inverted index

In [17]:
from typing import Iterator

d1 = "it is what it is"
d2 = "what is it"
d3 = "it is a banana"
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    yield ("{}".format(docid), document)

def MAP(docId:str, body:str):
  for word in set(body.split(' ')):
    yield (word, docId)

def REDUCE(word:str, docIds:Iterator[str]):
  yield (word, sorted(docIds))

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('what', ['0', '1']),
 ('it', ['0', '1', '2']),
 ('is', ['0', '1', '2']),
 ('a', ['2']),
 ('banana', ['2'])]

## WordCount

In [18]:
from typing import Iterator

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3]

def RECORDREADER():
  for (docid, document) in enumerate(documents):
    for (lineid, line) in enumerate(document.split('\n')):
      yield ("{}:{}".format(docid,lineid), line)

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

output = MapReduce(RECORDREADER, MAP, REDUCE)
output = list(output)
output

[('', 3), ('it', 9), ('is', 9), ('what', 5), ('a', 1), ('banana', 1)]

# MapReduce Distributed

Добавляется в модель фабрика RECORDREARER-ов --- INPUTFORMAT, функция распределения промежуточных результатов по партициям PARTITIONER, и функция COMBINER для частичной аггрегации промежуточных результатов до распределения по новым партициям.

In [19]:
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def groupbykey_distributed(map_partitions, PARTITIONER):
  global reducers
  partitions = [dict() for _ in range(reducers)]
  for map_partition in map_partitions:
    for (k2, v2) in map_partition:
      p = partitions[PARTITIONER(k2)]
      p[k2] = p.get(k2, []) + [v2]
  return [(partition_id, sorted(partition.items(), key=lambda x: x[0])) for (partition_id, partition) in enumerate(partitions)]

def PARTITIONER(obj):
  global reducers
  return hash(obj) % reducers

def MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, PARTITIONER=PARTITIONER, COMBINER=None):
  map_partitions = map(lambda record_reader: flatten(map(lambda k1v1: MAP(*k1v1), record_reader)), INPUTFORMAT())
  if COMBINER != None:
    map_partitions = map(lambda map_partition: flatten(map(lambda k2v2: COMBINER(*k2v2), groupbykey(map_partition))), map_partitions)
  reduce_partitions = groupbykey_distributed(map_partitions, PARTITIONER) # shuffle
  reduce_outputs = map(lambda reduce_partition: (reduce_partition[0], flatten(map(lambda reduce_input_group: REDUCE(*reduce_input_group), reduce_partition[1]))), reduce_partitions)

  print("{} key-value pairs were sent over a network.".format(sum([len(vs) for (k,vs) in flatten([partition for (partition_id, partition) in reduce_partitions])])))
  return reduce_outputs

## Спецификация MapReduce Distributed


```
f (k1, v1) -> (k2,v2)*
g (k2, v2*) -> (k3,v3)*

e1 (k1, v1)
e2 (k2, v2)
partition1 (k2, v2)*
partition2 (k2, v2*)*

flatmap (e1->e2*, e1*) -> partition1*
groupby (partition1*) -> partition2*

mapreduce ((k1,v1)*) -> (k3,v3)*
mapreduce .flatmap(f).groupby(k2).flatmap(g)
```



## WordCount

In [20]:
from typing import Iterator
import numpy as np

d1 = """
it is what it is
it is what it is
it is what it is"""
d2 = """
what is it
what is it"""
d3 = """
it is a banana"""
documents = [d1, d2, d3, d1, d2, d3]

maps = 3
reducers = 2

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for (docid, document) in enumerate(split):
      for (lineid, line) in enumerate(document.split('\n')):
        yield ("{}:{}".format(docid,lineid), line)

  split_size =  int(np.ceil(len(documents)/maps))
  for i in range(0, len(documents), split_size):
    yield RECORDREADER(documents[i:i+split_size])

def MAP(docId:str, line:str):
  for word in line.split(" "):
    yield (word, 1)

def REDUCE(word:str, counts:Iterator[int]):
  sum = 0
  for c in counts:
    sum += c
  yield (word, sum)

# try to set COMBINER=REDUCER and look at the number of values sent over the network
partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

56 key-value pairs were sent over a network.


[(0, [('', 6), ('banana', 2), ('what', 10)]),
 (1, [('a', 2), ('is', 18), ('it', 18)])]

## TeraSort

In [21]:
import numpy as np

input_values = np.random.rand(30)
maps = 3
reducers = 2
min_value = 0.0
max_value = 1.0

def INPUTFORMAT():
  global maps

  def RECORDREADER(split):
    for value in split:
        yield (value, None)

  split_size =  int(np.ceil(len(input_values)/maps))
  for i in range(0, len(input_values), split_size):
    yield RECORDREADER(input_values[i:i+split_size])

def MAP(value:int, _):
  yield (value, None)

def PARTITIONER(key):
  global reducers
  global max_value
  global min_value
  bucket_size = (max_value-min_value)/reducers
  bucket_id = 0
  while((key>(bucket_id+1)*bucket_size) and ((bucket_id+1)*bucket_size<max_value)):
    bucket_id += 1
  return bucket_id

def REDUCE(value:int, _):
  yield (None,value)

partitioned_output = MapReduceDistributed(INPUTFORMAT, MAP, REDUCE, COMBINER=None, PARTITIONER=PARTITIONER)
partitioned_output = [(partition_id, list(partition)) for (partition_id, partition) in partitioned_output]
partitioned_output

30 key-value pairs were sent over a network.


[(0,
  [(None, np.float64(0.06084678050308845)),
   (None, np.float64(0.11697741452811161)),
   (None, np.float64(0.15594536578817764)),
   (None, np.float64(0.1925667417501823)),
   (None, np.float64(0.2486460780582237)),
   (None, np.float64(0.30359876003543873)),
   (None, np.float64(0.3217013843453881)),
   (None, np.float64(0.3652727586793517)),
   (None, np.float64(0.3904318086059835)),
   (None, np.float64(0.4150319012520962)),
   (None, np.float64(0.4201832155298799)),
   (None, np.float64(0.46094382968059533)),
   (None, np.float64(0.47509342363986284)),
   (None, np.float64(0.4983070953451597))]),
 (1,
  [(None, np.float64(0.5276297301418337)),
   (None, np.float64(0.5736472884718805)),
   (None, np.float64(0.6909317297362514)),
   (None, np.float64(0.7014330811794891)),
   (None, np.float64(0.7111666998881764)),
   (None, np.float64(0.7258085386180217)),
   (None, np.float64(0.7582672264815197)),
   (None, np.float64(0.7620403889960228)),
   (None, np.float64(0.7664205407114

# Упражнения
Упражнения взяты из Rajaraman A., Ullman J. D. Mining of massive datasets. – Cambridge University Press, 2011.


Для выполнения заданий переопределите функции RECORDREADER, MAP, REDUCE. Для модели распределённой системы может потребоваться переопределение функций PARTITION и COMBINER.

### Максимальное значение ряда

Разработайте MapReduce алгоритм, который находит максимальное число входного списка чисел.

In [35]:
from multiprocessing import Pool
import math

def map_max(chunk):
    return max(chunk)

def reduce_max(results):
    return max(results)

def mapreduce_max(numbers, num_workers=4):

    chunk_size = math.ceil(len(numbers) / num_workers)
    chunks = [numbers[i:i + chunk_size] for i in range(0, len(numbers), chunk_size)]

    with Pool(num_workers) as pool:
        mapped = pool.map(map_max, chunks)

    return reduce_max(mapped)

data = [5, 8, 2, 15, 3, 21, 7, 9, 30, 4]

result = mapreduce_max(data)
print("Максимальное число:", result)

Максимальное число: 30


### Арифметическое среднее

Разработайте MapReduce алгоритм, который находит арифметическое среднее.

$$\overline{X} = \frac{1}{n}\sum_{i=0}^{n} x_i$$


In [37]:
from multiprocessing import Pool
import math

def map_mean(chunk):
    return sum(chunk), len(chunk)

def reduce_mean(results):
    total_sum = 0
    total_count = 0

    for s, c in results:
        total_sum += s
        total_count += c

    return total_sum / total_count

def mapreduce_mean(numbers, num_workers=4):
    chunk_size = math.ceil(len(numbers) / num_workers)
    chunks = [numbers[i:i+chunk_size] for i in range(0, len(numbers), chunk_size)]

    with Pool(num_workers) as pool:
        mapped = pool.map(map_mean, chunks)

    return reduce_mean(mapped)

data = [10, 20, 30, 40, 50, 60, 70, 80]

result = mapreduce_mean(data)

print("Среднее значение:", result)

Среднее значение: 45.0


### GroupByKey на основе сортировки

Реализуйте groupByKey на основе сортировки, проверьте его работу на примерах

In [38]:
def groupByKey_sort(pairs):
    pairs = sorted(pairs, key=lambda x: x[0])
    result = []
    current_key = None
    values = []

    for k, v in pairs:
        if k != current_key:
            if current_key is not None:
                result.append((current_key, values))
            current_key = k
            values = [v]
        else:
            values.append(v)

    if current_key is not None:
        result.append((current_key, values))

    return result


data1 = [("a",1),("b",2),("a",3),("b",4),("c",5)]
data2 = [(1,"x"),(2,"y"),(1,"z"),(3,"q"),(2,"w")]

print(groupByKey_sort(data1))
print(groupByKey_sort(data2))

[('a', [1, 3]), ('b', [2, 4]), ('c', [5])]
[(1, ['x', 'z']), (2, ['y', 'w']), (3, ['q'])]


### Drop duplicates (set construction, unique elements, distinct)

Реализуйте распределённую операцию исключения дубликатов

In [39]:
from multiprocessing import Pool
import math

def map_unique(chunk):
    return set(chunk)

def reduce_unique(results):
    result = set()
    for s in results:
        result |= s
    return list(result)

def distributed_distinct(data, workers=4):
    size = math.ceil(len(data) / workers)
    chunks = [data[i:i+size] for i in range(0, len(data), size)]
    with Pool(workers) as p:
        mapped = p.map(map_unique, chunks)
    return reduce_unique(mapped)

data = [1,2,3,2,4,1,5,3,6,4,7,5]

print(distributed_distinct(data))

[1, 2, 3, 4, 5, 6, 7]


#Операторы реляционной алгебры
### Selection (Выборка)

**The Map Function**: Для  каждого кортежа $t \in R$ вычисляется истинность предиката $C$. В случае истины создаётся пара ключ-значение $(t, t)$. В паре ключ и значение одинаковы, равны $t$.

**The Reduce Function:** Роль функции Reduce выполняет функция идентичности, которая возвращает то же значение, что получила на вход.



In [42]:
from collections import defaultdict

def map_function(R, C):
    result = []
    for t in R:
        if C(t):
            result.append((t, t))
    return result

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_identity(grouped):
    result = []
    for k, values in grouped:
        for v in values:
            result.append(v)
    return result

def mapreduce_select(R, C):
    mapped = map_function(R, C)
    grouped = shuffle(mapped)
    reduced = reduce_identity(grouped)
    return reduced

R = [1,2,3,4,5,6,7,8]
C = lambda t: t % 2 == 0

print(mapreduce_select(R, C))

[2, 4, 6, 8]


### Projection (Проекция)

Проекция на множество атрибутов $S$.

**The Map Function:** Для каждого кортежа $t \in R$ создайте кортеж $t′$, исключая  из $t$ те значения, атрибуты которых не принадлежат  $S$. Верните пару $(t′, t′)$.

**The Reduce Function:** Для каждого ключа $t′$, созданного любой Map задачей, вы получаете одну или несколько пар $(t′, t′)$. Reduce функция преобразует $(t′, [t′, t′, . . . , t′])$ в $(t′, t′)$, так, что для ключа $t′$ возвращается одна пара  $(t′, t′)$.

In [44]:
from collections import defaultdict

def map_function(R, S):
    result = []
    for t in R:
        t_prime = tuple(t[i] for i in S)
        result.append((t_prime, t_prime))
    return result

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped):
    result = []
    for k, values in grouped:
        result.append((k, k))
    return result

def mapreduce_projection(R, S):
    mapped = map_function(R, S)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

R = [
    (1, "Андрей", 20),
    (2, "Миша", 21),
    (3, "Андрей", 20),
    (4, "Вика", 22)
]

S = [1, 2]

print(mapreduce_projection(R, S))

[(('Андрей', 20), ('Андрей', 20)), (('Миша', 21), ('Миша', 21)), (('Вика', 22), ('Вика', 22))]


### Union (Объединение)

**The Map Function:** Превратите каждый входной кортеж $t$ в пару ключ-значение $(t, t)$.

**The Reduce Function:** С каждым ключом $t$ будет ассоциировано одно или два значений. В обоих случаях создайте $(t, t)$ в качестве выходного значения.

In [45]:
from collections import defaultdict

def map_function(R):
    return [(t, t) for t in R]

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped):
    result = []
    for k, values in grouped:
        result.append((k, k))
    return result

def mapreduce_operation(R):
    mapped = map_function(R)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

R = [(1,"A"), (2,"B"), (1,"A"), (3,"C")]

print(mapreduce_operation(R))

[((1, 'A'), (1, 'A')), ((2, 'B'), (2, 'B')), ((3, 'C'), (3, 'C'))]


### Intersection (Пересечение)

**The Map Function:** Превратите каждый кортеж $t$ в пары ключ-значение $(t, t)$.

**The Reduce Function:** Если для ключа $t$ есть список из двух элементов $[t, t]$ $-$ создайте пару $(t, t)$. Иначе, ничего не создавайте.

In [46]:
from collections import defaultdict

def map_function(R):
    return [(t, t) for t in R]

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped):
    result = []
    for k, values in grouped:
        if len(values) == 2:
            result.append((k, k))
    return result

def mapreduce_operation(R):
    mapped = map_function(R)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

R = [(1,"A"), (2,"B"), (1,"A"), (3,"C"), (2,"B"), (4,"D")]

print(mapreduce_operation(R))

[((1, 'A'), (1, 'A')), ((2, 'B'), (2, 'B'))]


### Difference (Разница)

**The Map Function:** Для кортежа $t \in R$, создайте пару $(t, R)$, и для кортежа $t \in S$, создайте пару $(t, S)$. Задумка заключается в том, чтобы значение пары было именем отношения $R$ or $S$, которому принадлежит кортеж (а лучше, единичный бит, по которому можно два отношения различить $R$ or $S$), а не весь набор атрибутов отношения.

**The Reduce Function:** Для каждого ключа $t$, если соответствующее значение является списком $[R]$, создайте пару $(t, t)$. В иных случаях не предпринимайте действий.

In [47]:
from collections import defaultdict

def map_function(R, S):
    result = []
    for t in R:
        result.append((t, "R"))
    for t in S:
        result.append((t, "S"))
    return result

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped):
    result = []
    for k, values in grouped:
        if values == ["R"]:
            result.append((k, k))
    return result

def mapreduce_semi_join(R, S):
    mapped = map_function(R, S)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

R = [(1,"A"), (2,"B"), (3,"C")]
S = [(2,"B"), (4,"D")]

print(mapreduce_semi_join(R, S))

[((1, 'A'), (1, 'A')), ((3, 'C'), (3, 'C'))]


### Natural Join

**The Map Function:** Для каждого кортежа $(a, b)$ отношения $R$, создайте пару $(b,(R, a))$. Для каждого кортежа $(b, c)$ отношения $S$, создайте пару $(b,(S, c))$.

**The Reduce Function:** Каждый ключ $b$ будет асоциирован со списком пар, которые принимают форму либо $(R, a)$, либо $(S, c)$. Создайте все пары, одни, состоящие из  первого компонента $R$, а другие, из первого компонента $S$, то есть $(R, a)$ и $(S, c)$. На выходе вы получаете последовательность пар ключ-значение из списков ключей и значений. Ключ не нужен. Каждое значение, это тройка $(a, b, c)$ такая, что $(R, a)$ и $(S, c)$ это принадлежат входному списку значений.

In [48]:
from collections import defaultdict
from itertools import product

def map_function(R, S):
    mapped = []
    for a, b in R:
        mapped.append((b, ("R", a)))
    for b, c in S:
        mapped.append((b, ("S", c)))
    return mapped

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped):
    result = []
    for k, values in grouped:
        R_values = [a for typ, a in values if typ == "R"]
        S_values = [c for typ, c in values if typ == "S"]
        for a, c in product(R_values, S_values):
            result.append((a, k, c))
    return result

def mapreduce_join(R, S):
    mapped = map_function(R, S)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

R = [(1, 2), (2, 3), (3, 4)]
S = [(2, 20), (3, 30), (3, 35), (4, 40)]

print(mapreduce_join(R, S))

[(1, 2, 20), (2, 3, 30), (2, 3, 35), (3, 4, 40)]


### Grouping and Aggregation (Группировка и аггрегация)

**The Map Function:** Для каждого кортежа $(a, b, c$) создайте пару $(a, b)$.

**The Reduce Function:** Ключ представляет ту или иную группу. Примение аггрегирующую операцию $\theta$ к списку значений $[b1, b2, . . . , bn]$ ассоциированных с ключом $a$. Возвращайте в выходной поток $(a, x)$, где $x$ результат применения  $\theta$ к списку. Например, если $\theta$ это $SUM$, тогда $x = b1 + b2 + · · · + bn$, а если $\theta$ is $MAX$, тогда $x$ это максимальное из значений $b1, b2, . . . , bn$.

In [49]:
from collections import defaultdict

def map_function(data):
    mapped = []
    for a, b, c in data:
        mapped.append((a, b))
    return mapped

def shuffle(mapped):
    grouped = defaultdict(list)
    for k, v in mapped:
        grouped[k].append(v)
    return grouped.items()

def reduce_function(grouped, theta):
    result = []
    for k, values in grouped:
        if theta == "SUM":
            agg = sum(values)
        elif theta == "MAX":
            agg = max(values)
        elif theta == "MIN":
            agg = min(values)
        else:
            raise ValueError("Unsupported aggregation function")
        result.append((k, agg))
    return result

def mapreduce_aggregate(data, theta):
    mapped = map_function(data)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped, theta)
    return reduced

data = [(1, 10, 100), (2, 5, 200), (1, 20, 150), (2, 15, 250), (3, 7, 300)]

print("SUM:", mapreduce_aggregate(data, "SUM"))
print("MAX:", mapreduce_aggregate(data, "MAX"))

SUM: [(1, 30), (2, 20), (3, 7)]
MAX: [(1, 20), (2, 15), (3, 7)]


#

### Matrix-Vector multiplication

Случай, когда вектор не помещается в памяти Map задачи


In [50]:
from collections import defaultdict

def map_function(matrix_rows, vector_provider):
    mapped = []
    for i, row in matrix_rows:
        for j, a_ij in enumerate(row):
            v_j = vector_provider(j)
            mapped.append((i, a_ij * v_j))
    return mapped

def shuffle(mapped):
    grouped = defaultdict(list)
    for i, value in mapped:
        grouped[i].append(value)
    return grouped.items()

def reduce_function(grouped):
    result = {}
    for i, values in grouped:
        result[i] = sum(values)
    return result

def mapreduce_matrix_vector(matrix_rows, vector_provider):
    mapped = map_function(matrix_rows, vector_provider)
    grouped = shuffle(mapped)
    reduced = reduce_function(grouped)
    return reduced

matrix_rows = [
    (0, [1, 2, 3]),
    (1, [4, 5, 6]),
    (2, [7, 8, 9])
]

vector = [1, 0, 1]
def vector_provider(j):
    return vector[j]

result = mapreduce_matrix_vector(matrix_rows, vector_provider)
print(result)

{0: 4, 1: 10, 2: 16}


## Matrix multiplication (Перемножение матриц)

Если у нас есть матрица $M$ с элементами $m_{ij}$ в строке $i$ и столбце $j$, и матрица $N$ с элементами $n_{jk}$ в строке $j$ и столбце $k$, тогда их произведение $P = MN$ есть матрица $P$ с элементами $p_{ik}$ в строке $i$ и столбце $k$, где

$$p_{ik} =\sum_{j} m_{ij}n_{jk}$$

Необходимым требованием является одинаковое количество столбцов в $M$ и строк в $N$, чтобы операция суммирования по  $j$ была осмысленной. Мы можем размышлять о матрице, как об отношении с тремя атрибутами: номер строки, номер столбца, само значение. Таким образом матрица $M$ предстваляется как отношение $ M(I, J, V )$, с кортежами $(i, j, m_{ij})$, и, аналогично, матрица $N$ представляется как отношение $N(J, K, W)$, с кортежами $(j, k, n_{jk})$. Так как большие матрицы как правило разреженные (большинство значений равно 0), и так как мы можем нулевыми значениями пренебречь (не хранить), такое реляционное представление достаточно эффективно для больших матриц. Однако, возможно, что координаты $i$, $j$, и $k$ неявно закодированы в смещение позиции элемента относительно начала файла, вместо явного хранения. Тогда, функция Map (или Reader) должна быть разработана таким образом, чтобы реконструировать компоненты $I$, $J$, и $K$ кортежей из смещения.

Произведение $MN$ это фактически join, за которым следуют группировка по ключу и аггрегация. Таким образом join отношений $M(I, J, V )$ и $N(J, K, W)$, имеющих общим только атрибут $J$, создаст кортежи $(i, j, k, v, w)$ из каждого кортежа $(i, j, v) \in M$ и кортежа $(j, k, w) \in N$. Такой 5 компонентный кортеж представляет пару элементов матрицы $(m_{ij} , n_{jk})$. Что нам хотелось бы получить на самом деле, это произведение этих элементов, то есть, 4 компонентный кортеж$(i, j, k, v \times w)$, так как он представляет произведение $m_{ij}n_{jk}$. Мы представляем отношение как результат одной MapReduce операции, в которой мы можем произвести группировку и аггрегацию, с $I$ и $K$  атрибутами, по которым идёт группировка, и суммой  $V \times W$.





In [22]:
# MapReduce model
def flatten(nested_iterable):
  for iterable in nested_iterable:
    for element in iterable:
      yield element

def groupbykey(iterable):
  t = {}
  for (k2, v2) in iterable:
    t[k2] = t.get(k2, []) + [v2]
  return t.items()

def MapReduce(RECORDREADER, MAP, REDUCE):
  return flatten(map(lambda x: REDUCE(*x), groupbykey(flatten(map(lambda x: MAP(*x), RECORDREADER())))))

Реализуйте перемножение матриц с использованием модельного кода MapReduce для одной машины в случае, когда одна матрица хранится в памяти, а другая генерируется RECORDREADER-ом.

In [62]:
import numpy as np
from collections import defaultdict

M = np.array([[1,2,3],
              [4,5,6]])

def MAP_MN(source, key, value):
    if source == "M":
        i,j = key
        yield (j, ("M", i, value))
    else:
        j,k = key
        yield (j, ("N", k, value))

def REDUCE_MN(key, values):
    M_vals = []
    N_vals = []
    for tag, idx, val in values:
        if tag == "M":
            M_vals.append((idx, val))
        else:
            N_vals.append((idx, val))
    for i, m in M_vals:
        for k, n in N_vals:
            yield ((i,k), m*n)

def RECORDREADER():
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            yield ("M", (i,j), M[i,j])

    N = np.array([[7,8],
                  [9,10],
                  [11,12]])
    for j in range(N.shape[0]):
        for k in range(N.shape[1]):
            yield ("N", (j,k), N[j,k])

result = MapReduce(RECORDREADER, MAP_MN, REDUCE_MN)

C = defaultdict(int)
for (i,k), val in result:
    C[(i,k)] += val

C_array = np.zeros((M.shape[0], 2))
for (i,k), val in C.items():
    C_array[i,k] = val

print("MapReduce результат:\n", C_array)
print("NumPy результат:\n", M @ np.array([[7,8],[9,10],[11,12]]))

MapReduce результат:
 [[ 58.  64.]
 [139. 154.]]
NumPy результат:
 [[ 58  64]
 [139 154]]


Проверьте своё решение

In [68]:
# Проверка с NumPy
M_np = np.array([[1,2,3],
                 [4,5,6]])
N_np = np.array([[7,8],
                 [9,10],
                 [11,12]])
C_np = M_np @ N_np

print("MapReduce результат:\n", C_array)
print("NumPy результат:\n", C_np)
print("Сравнение:", np.allclose(C_array, C_np))

MapReduce результат:
 [[ 58.  64.]
 [139. 154.]]
NumPy результат:
 [[ 58  64]
 [139 154]]
Сравнение: True


Реализуйте перемножение матриц  с использованием модельного кода MapReduce для одной машины в случае, когда обе матрицы генерируются в RECORDREADER. Например, сначала одна, а потом другая.

In [69]:
def RECORDREADER():
  for i in range(I):
    for j in range(J):
      yield ("M", (i,j), small_mat[i,j])

  for j in range(J):
    for k in range(K):
      yield ("N", (j,k), big_mat[j,k])


def MAP(source, key, value):
  if source == "M":
    i, j = key
    yield (j, ("M", i, value))
  else:
    j, k = key
    yield (j, ("N", k, value))


def REDUCE(key, values):
  M_vals = []
  N_vals = []

  for v in values:
    if v[0] == "M":
      M_vals.append(v[1:])
    else:
      N_vals.append(v[1:])

  for (i, m) in M_vals:
    for (k, n) in N_vals:
      yield ((i,k), m*n)

Реализуйте перемножение матриц с использованием модельного кода MapReduce Distributed, когда каждая матрица генерируется в своём RECORDREADER.

In [70]:
def MAP(source, key, value):
  if source == "M":
    i, j = key
    yield (j, ("M", i, value))
  else:
    j, k = key
    yield (j, ("N", k, value))


def REDUCE(key, values):
  M_vals = []
  N_vals = []

  for tag, idx, val in values:
    if tag == "M":
      M_vals.append((idx, val))
    else:
      N_vals.append((idx, val))

  for i, m in M_vals:
    for k, n in N_vals:
      yield ((i,k), m*n)

Обобщите предыдущее решение на случай, когда каждая матрица генерируется несколькими RECORDREADER-ами, и проверьте его работоспособность. Будет ли работать решение, если RECORDREADER-ы будут генерировать случайное подмножество элементов матрицы?

In [71]:
from collections import defaultdict

def MAP(source, key, value):
    if source == "M":
        i, j = key
        yield (j, ("M", i, value))
    else:
        j, k = key
        yield (j, ("N", k, value))

def REDUCE(key, values):
    M_vals = []
    N_vals = []

    for tag, idx, val in values:
        if tag == "M":
            M_vals.append((idx, val))
        else:
            N_vals.append((idx, val))

    for i, m in M_vals:
        for k, n in N_vals:
            yield ((i,k), m*n)

def shuffle(mapped_list):
    grouped = defaultdict(list)
    for mapped in mapped_list:
        for key, value in mapped:
            grouped[key].append(value)
    return grouped.items()

def mapreduce_matrix_multiply_distributed(M_readers, N_readers):
    mapped = []
    for reader in M_readers:
        for key, value in reader():
            mapped.extend(MAP("M", key, value))
    for reader in N_readers:
        for key, value in reader():
            mapped.extend(MAP("N", key, value))

    grouped = shuffle([mapped])
    reduced = []
    for key, values in grouped:
        reduced.extend(REDUCE(key, values))

    # Суммируем дублирующие ключи
    result = defaultdict(int)
    for (i,k), val in reduced:
        result[(i,k)] += val
    return result


import numpy as np
# Матрица M 2x3
M = np.array([[1,2,3],
              [4,5,6]])
# Матрица N 3x2
N = np.array([[7,8],
              [9,10],
              [11,12]])

# Генераторы RecordReader-ов для M
def M_reader1():
    yield (0,0), M[0,0]
    yield (0,1), M[0,1]

def M_reader2():
    yield (0,2), M[0,2]
    yield (1,0), M[1,0]
    yield (1,1), M[1,1]
    yield (1,2), M[1,2]

# Генераторы RecordReader-ов для N
def N_reader1():
    yield (0,0), N[0,0]
    yield (0,1), N[0,1]

def N_reader2():
    yield (1,0), N[1,0]
    yield (1,1), N[1,1]
    yield (2,0), N[2,0]
    yield (2,1), N[2,1]

result = mapreduce_matrix_multiply_distributed(
    M_readers=[M_reader1, M_reader2],
    N_readers=[N_reader1, N_reader2]
)

C = np.zeros((M.shape[0], N.shape[1]))
for (i,k), value in result.items():
    C[i,k] = value

print("MapReduce результат:\n", C)
print("NumPy результат:\n", M @ N)
print("Сравнение:", np.allclose(C, M @ N))

MapReduce результат:
 [[ 58.  64.]
 [139. 154.]]
NumPy результат:
 [[ 58  64]
 [139 154]]
Сравнение: True


Будет ли работать, если RECORDREADER генерирует случайное подмножество?
Нет, если подмножества неполные, потому что произведение требует все пары m_ij * n_jk